# Pipeline

## Imports

In [6]:
import numpy as np
import pandas as pd
import os
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import torch

import resnet as features
import data_utils

TRAIN_CSV = "../../data/tabular/train/train.csv"
TEST_CSV = "../../data/tabular/test/test.csv"
IMAGE_FOLDER = "../../data/image" 

## Load and preprocess data with ResNet50 to extract features

In [ ]:
print("Loading Training Data...")
df_train, y_train = data_utils.load_train_data(TRAIN_CSV)
print(f"   Found {len(df_train)} unique training images.")

print("Extracting Features (ResNet)...")
# Check if we have a GPU, otherwise CPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
extractor = features.ResNetExtractor(device=device)

def get_full_path(p): 
    return os.path.join(IMAGE_FOLDER, p)

# Extract features for training images
# This might take a few minutes depending on your hardware
train_feats = []
for idx, path in enumerate(df_train['image_path']):
    if idx % 50 == 0: print(f"   Processed {idx}/{len(df_train)}...")
    train_feats.append(extractor.get_features(get_full_path(path)))

X_train_raw = np.array(train_feats)

Loading Training Data...
   Found 357 unique training images.
2. Extracting Features (ResNet)...
Loading ResNet50 on cpu...


/Users/victorslorer/anaconda3/envs/INF265/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/victorslorer/anaconda3/envs/INF265/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


   Processed 0/357...
   Processed 50/357...
   Processed 100/357...
   Processed 150/357...
   Processed 200/357...
   Processed 250/357...
   Processed 300/357...
   Processed 350/357...


## PCA for dimensionality reduction

In [10]:
print("Reducing Dimensions (PCA)...")
# Reduce 2048 features -> 50 features
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_raw)

Reducing Dimensions (PCA)...


## Prediction with regression forest

In [11]:
print("4. Training Random Forest (Multi-Output)...")
# Split into internal train/val set to check performance
X_tr, X_val, y_tr, y_val = train_test_split(X_train_pca, y_train, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)

# Evaluation
score = rf.score(X_val, y_val)
preds_val = rf.predict(X_val)
# RMSE for Total Biomass (Index 0 is Dry_Total_g)
rmse = np.sqrt(mean_squared_error(y_val[:, 0], preds_val[:, 0]))

print(f"   Validation R^2 Score: {score:.4f}")
print(f"   Validation RMSE (Total Biomass): {rmse:.2f}")

4. Training Random Forest (Multi-Output)...
   Validation R^2 Score: 0.2663
   Validation RMSE (Total Biomass): 20.02
